# QLoRA NER pipeline — DisTEMIST → CardioCCC with Ministral-3-8B-Instruct-2512

Train on the original 750 DisTEMIST training documents and evaluate on the 250 CardioCCC test documents.
The original training settings, Spanish prompt, sentence segmentation and decoding are preserved.
CardioCCC is used only for external inference and evaluation, with no prompt or parameter tuning.

## Contents

1. [Environment and dependencies](#1-environment-and-dependencies)
2. [Configuration](#2-configuration)
3. [Model loading](#3-model-loading)
4. [LoRA adapters](#4-lora-adapters)
5. [Dataset](#5-dataset)
6. [Training](#6-training)
7. [Inference preparation](#7-inference-preparation)
8. [Inference](#8-inference)
9. [Evaluation](#9-evaluation)


## 1. Environment and dependencies

Pin the recorded PyTorch version and source snapshots from the original execution date.
The complete installed environment is saved with the results.


In [ ]:
import subprocess
import sys

def pip_install(*arguments):
    subprocess.run([sys.executable, "-m", "pip", *arguments], check=True)

# Kaggle preloads Pillow's native extension; keep that installed version in this session.
pip_install("install", "torch==2.10.0", "torchvision==0.25.0", "torchaudio==2.10.0",
            "--index-url", "https://download.pytorch.org/whl/cu128", "--quiet")
# These source snapshots reconstruct the original date; its Git commits were not recorded.
pip_install("install", "--no-deps",
            "unsloth @ https://github.com/unslothai/unsloth/archive/556f396b3cd63da525e2cdac1a5f1c606f4494ce.zip",
            "unsloth_zoo @ https://github.com/unslothai/unsloth-zoo/archive/606d2b2d7a55129cebfc6b3a044d1aad568c1d3f.zip", "--quiet")
pip_install("install", "peft==0.19.1", "accelerate==1.13.0", "bitsandbytes==0.49.2",
            "trl==0.15.2", "datasets==4.8.5", "huggingface-hub==1.16.4", "tokenizers==0.22.2",
            "torchao==0.17.0", "tyro==1.0.13", "hf-transfer==0.1.9", "msgspec==0.21.1",
            "cut-cross-entropy==25.1.1", "diffusers==0.38.0", "spacy==3.8.14", "--quiet")
subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_md", "--quiet"], check=True)
# Install this snapshot last so dependency resolution cannot replace it with a newer release.
pip_install("install", "--no-deps", "transformers @ https://github.com/huggingface/transformers/archive/9ded3dbbfc19d828c719d8ddfd565311314bec26.zip", "--quiet")
import importlib.metadata
environment = {p: importlib.metadata.version(p) for p in [
    "torch", "transformers", "tokenizers", "unsloth", "unsloth_zoo", "trl", "Pillow",
]}
if environment["transformers"] != "5.10.0.dev0" or environment["tokenizers"] != "0.22.2":
    raise RuntimeError(f"The installed environment differs from the selected source snapshots: {environment}")
print("Environment:", environment, flush=True)


## 2. Configuration

Hyperparameters, verified dataset paths and credentials.


In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_NAME = "unsloth/Ministral-3-8B-Instruct-2512-unsloth-bnb-4bit"
MODEL_REVISION = "20b41a4990a9eac0ffafa76478c3a56859391156"

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R         = 64
LORA_ALPHA     = 128
LORA_DROPOUT   = 0
MAX_SEQ_LENGTH = 4096

# ── Training ─────────────────────────────────────────────────────────────
BATCH_SIZE       = 1
GRAD_ACCUM       = 32
LEARNING_RATE    = 3e-5
NUM_TRAIN_EPOCHS = 2
WARMUP_RATIO     = 0.05
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 0.3
TRAIN_SEED       = 3407
OUTPUT_DIR       = "outputs_lora"

# ── Dataset / chunking ────────────────────────────────────────────────────────
MAX_CHUNK_CHARS      = None   # None means one sentence per chunk
SPACY_MODEL          = "es_core_news_md"
MAX_NEGATIVE_RATIO   = None   # Example: 0.5 means one negative per two positives
NEGATIVE_SAMPLE_SEED = 3407

# ── System prompt ────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Actua como un sistema NER medico de alta precision.\n\n"
    "REGLAS DE EXTRACCION:\n"
    "1. Extrae exclusivamente entidades de tipo ENFERMEDAD.\n"
    "2. COPIA Y PEGA de forma literal: No cambies mayusculas, minusculas ni tildes.\n"
    "3. PROHIBIDO USAR SINONIMOS: Si el texto dice 'neoplasia', no escribas 'cancer'.\n"
    "4. REPETICIONES: Si una enfermedad aparece varias veces en el texto, debes listarla varias veces en lineas separadas.\n"
    "5. ORDEN: Extrae las menciones en el mismo orden en que aparecen en el texto.\n"
    "6. FORMATO: Únicamente el texto de la mención, una por línea. PROHIBIDO usar caracteres de lista al inicio (como '-', '*', '•', '1.').\n"
    "7. Si no hay nada, devuelve un texto vacio."
)

# ── Inference ────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS   = 300
MAX_TEST_FILES   = 250   # None means all files
PRINT_RAW_OUTPUT = True

# ── Dataset paths ────────────────────────────────────────────────────────────
import csv
import hashlib
import json
from zipfile import ZipFile

DATA_ROOT = Path("/kaggle/working/data")
manifests = sorted(Path("/kaggle/input").rglob("dataset_manifest.json"))
archives = sorted(Path("/kaggle/input").rglob("distemist_cardioccc_ministral.zip"))
if len(manifests) == 1:
    DATA_ROOT = manifests[0].parent
elif not manifests and len(archives) == 1:
    with ZipFile(archives[0]) as archive:
        if any(Path(name).is_absolute() or ".." in Path(name).parts for name in archive.namelist()):
            raise ValueError("Unsafe path in the dataset archive.")
        archive.extractall(DATA_ROOT)
else:
    raise FileNotFoundError("Attach exactly one copy of the DisTEMIST–CardioCCC Ministral dataset.")

dataset_manifest = json.loads((DATA_ROOT / "dataset_manifest.json").read_text(encoding="utf-8"))
DATASET_FINGERPRINT = "ef8dd7e9b2aa3258144c377a22a1ce9c65e269a9572ebe0f90efb64315207113"
fingerprint = hashlib.sha256(json.dumps(dataset_manifest["files"], sort_keys=True, separators=(",", ":")).encode()).hexdigest()
if fingerprint != DATASET_FINGERPRINT or dataset_manifest["fingerprint"] != fingerprint:
    raise ValueError("The dataset manifest differs from the frozen experiment.")
for relative, expected in dataset_manifest["files"].items():
    if Path(relative).is_absolute() or ".." in Path(relative).parts:
        raise ValueError("Unsafe path in the dataset manifest.")
    if hashlib.sha256((DATA_ROOT / relative).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Dataset checksum mismatch: {relative}")

TEXT_FILES_TRAIN_DIR = str(DATA_ROOT / "distemist/text_files_train")
TEXT_FILES_TEST_DIR = str(DATA_ROOT / "cardioccc/test/texts")
GS_TRAIN_TSV = str(DATA_ROOT / "distemist/distemist_subtrack1_training_mentions.tsv")
GS_TEST_TSV = str(DATA_ROOT / "cardioccc/test/gold.tsv")
PREDICTIONS_TSV = "/kaggle/working/cardioccc_llm_predictions.tsv"
EVAL_SUMMARY_JSON = "/kaggle/working/evaluation_summary.json"
LORA_SAVE_DIR = "/kaggle/working/lora_adapter"

for directory, gold_path, expected_documents, expected_mentions in [
    (TEXT_FILES_TRAIN_DIR, GS_TRAIN_TSV, 750, 8065),
    (TEXT_FILES_TEST_DIR, GS_TEST_TSV, 250, 7884),
]:
    texts = {p.stem: p.read_bytes().decode("utf-8") for p in Path(directory).glob("*.txt")}
    with open(gold_path, encoding="utf-8", newline="") as stream:
        gold = list(csv.DictReader(stream, delimiter="\t"))
    if len(texts) != expected_documents or len(gold) != expected_mentions or {r["filename"] for r in gold} != set(texts):
        raise ValueError(f"Unexpected dataset counts in {directory}")
    for row in gold:
        start, end = int(row["off0"]), int(row["off1"])
        if row["label"] != "ENFERMEDAD" or not 0 <= start < end <= len(texts[row["filename"]]) or texts[row["filename"]][start:end] != row["span"]:
            raise ValueError(f"Invalid annotation in {gold_path}: {row['filename']}")
print("Verified dataset:", dataset_manifest["counts"])

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

print(f"Base model        : {MODEL_NAME}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"MAX_SEQ_LENGTH     : {MAX_SEQ_LENGTH}")
print(f"MAX_CHUNK_CHARS    : {MAX_CHUNK_CHARS}")
print(f"Epochs             : {NUM_TRAIN_EPOCHS}")
print(f"MAX_NEW_TOKENS     : {MAX_NEW_TOKENS}")
print(f"MAX_TEST_FILES     : {MAX_TEST_FILES}")
print(f"MAX_NEGATIVE_RATIO : {MAX_NEGATIVE_RATIO}")

## 3. Model loading

Load the same base model in 4-bit precision and record its revision and runtime environment.


In [ ]:
import torch
from unsloth import FastLanguageModel

print(f"Available GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {round(props.total_memory / 1024**3, 1)} GB VRAM")

if torch.cuda.device_count() != 2:
    raise RuntimeError("Select GPU T4 x2 before running this notebook.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.float16,
    load_in_4bit   = True,
    token          = hf_token,
    revision       = MODEL_REVISION,
)

if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

print(f"\nLoaded model: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
import importlib.metadata
import platform
import subprocess
import sys
from datetime import datetime, timezone

runtime = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": MODEL_NAME, "model_revision": MODEL_REVISION,
    "tokenizer_source_revision": MODEL_REVISION,
    "original_revision_status": "Not recorded by the original execution; the current HF revision is pinned for this new run.",
    "dataset_fingerprint": DATASET_FINGERPRINT,
    "training_corpus": "DisTEMIST train: 750 documents / 8,065 mentions",
    "external_test": "CardioCCC test: 250 documents / 7,884 mentions",
    "python": platform.python_version(), "cuda": torch.version.cuda,
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "software": {p: importlib.metadata.version(p) for p in [
        "torch", "transformers", "unsloth", "unsloth_zoo", "trl", "peft",
        "accelerate", "bitsandbytes", "datasets", "spacy", "es_core_news_md",
    ]},
    "source_notebook": "https://github.com/Honricris/Trabajo_Fin_Master/blob/21216f4f739f17cde890c0d51bc4fa23824b50e0/Decoders/Distemist/SFT/ministral-3-8b-instruct-sft(79.25).ipynb",
    "source_notebook_sha256": "478c0773cc0c2a82bac41190f7127aa4120d8cd662ae84b3c233a6ea4c7adbc9",
}
Path("/kaggle/working/runtime_settings.json").write_text(json.dumps(runtime, indent=2) + "\n")
Path("/kaggle/working/dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2) + "\n")
frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True)
Path("/kaggle/working/package_versions.txt").write_text(frozen.stdout)
print("RUNTIME " + json.dumps(runtime), flush=True)


## 4. LoRA adapters

Apply LoRA adapters to all linear projections, preserving the original configuration.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = "all-linear",
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,}  ({100*trainable/total:.2f}%)")
model.print_trainable_parameters()

## 5. Dataset

Sentence segmentation, Spanish training prompts and the original DisTEMIST training split.


In [ ]:
import random
import numpy as np
import pandas as pd
import spacy
from pathlib import Path
from datasets import Dataset

nlp_spacy = spacy.load(SPACY_MODEL, disable=["ner", "lemmatizer"])
print(f"spaCy: {nlp_spacy.meta['name']} v{nlp_spacy.meta['version']}")


def chunk_text_by_sentences(text, nlp, max_chars=MAX_CHUNK_CHARS):
    """Split text into complete-sentence chunks with character offsets.
    If max_chars is None, return one sentence per chunk."""
    doc = nlp(text)
    if max_chars is None:
        return [(text[s.start_char:s.end_char], s.start_char, s.end_char) for s in doc.sents]

    chunks, current_chars, chunk_start, current_end = [], 0, None, None
    for sent in doc.sents:
        if current_chars + len(sent.text) > max_chars and chunk_start is not None:
            chunks.append((text[chunk_start:current_end], chunk_start, current_end))
            current_chars, chunk_start, current_end = 0, None, None
        if chunk_start is None:
            chunk_start = sent.start_char
        current_chars += len(sent.text)
        current_end = sent.end_char
    if chunk_start is not None:
        chunks.append((text[chunk_start:current_end], chunk_start, current_end))
    return chunks


def mentions_in_chunk(spans_with_offsets, chunk_start, chunk_end):
    return [span for span, off0, off1 in spans_with_offsets if chunk_start <= off0 < chunk_end]


def build_training_example(text, mentions):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Texto para analizar:\n{text}"},
        {"role": "assistant", "content": "\n".join(mentions) if mentions else ""},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


df_gs = pd.read_csv(GS_TRAIN_TSV, sep="\t").sort_values(["filename", "off0"])
text_dir = Path(TEXT_FILES_TRAIN_DIR)

positive_texts, negative_texts = [], []
for doc_id, group in df_gs.groupby("filename", sort=False):
    txt_path = text_dir / f"{doc_id}.txt"
    if not txt_path.exists():
        continue
    text = txt_path.read_text(encoding="utf-8")
    spans = list(zip(group["span"], group["off0"], group["off1"]))
    chunks = chunk_text_by_sentences(text, nlp_spacy)
    changed = True
    while changed:
        changed = False
        for i in range(len(chunks) - 1):
            _, cs, ce = chunks[i]
            _, ns, ne = chunks[i + 1]
            for _, off0, off1 in spans:
                if cs <= off0 < ce and off1 > ce:
                    chunks[i] = (text[cs:ne], cs, ne)
                    chunks.pop(i + 1)
                    changed = True
                    break
            if changed:
                break
    for chunk_text, char_start, char_end in chunks:
        mentions = mentions_in_chunk(spans, char_start, char_end)
        example  = build_training_example(chunk_text, mentions)
        (positive_texts if mentions else negative_texts).append(example)

if MAX_NEGATIVE_RATIO is None:
    negative_texts_used = negative_texts
else:
    max_neg = max(0, int(len(positive_texts) * MAX_NEGATIVE_RATIO))
    negative_texts_used = (
        random.Random(NEGATIVE_SAMPLE_SEED).sample(negative_texts, k=max_neg)
        if max_neg < len(negative_texts) else negative_texts
    )

formatted_texts = positive_texts + negative_texts_used
dataset = Dataset.from_dict({"text": formatted_texts})

lengths   = [len(tokenizer.encode(t)) for t in formatted_texts]
truncados = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)

print(f"Dataset: {len(dataset)} chunks (max_chars={MAX_CHUNK_CHARS})")
print(f"Positive: {len(positive_texts)} | Negative used: {len(negative_texts_used)} / {len(negative_texts)} (ratio={MAX_NEGATIVE_RATIO})")
print(f"Tokens — min:{min(lengths)}  max:{max(lengths)}  mean:{np.mean(lengths):.0f}  p95:{np.percentile(lengths,95):.0f}")
print(f"Truncated (>{MAX_SEQ_LENGTH}): {truncados} ({100*truncados/len(lengths):.1f}%)")
print(f"Total gold mentions: {len(df_gs)}")

## 6. Training

Use `SFTTrainer` and `DataCollatorForCompletionOnlyLM` to train on assistant responses only.
Save the final adapter, tokenizer and training configuration before external inference.


In [ ]:
import gc
import torch
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"

tokenizer.padding_side = "right"

def filter_fn(examples):
    tokens = tokenizer(examples["text"], add_special_tokens=False)
    return [len(t) <= MAX_SEQ_LENGTH for t in tokens["input_ids"]]

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

train_data = (
    dataset
    .filter(filter_fn, batched=True, num_proc=2)
    .map(tokenize_fn, batched=True, num_proc=2, remove_columns=["text"])
)
print(f"Train: {len(train_data)} examples")

response_template_ids = tokenizer.encode("[/INST]", add_special_tokens=False)
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_data,
    data_collator = data_collator,
    args = SFTConfig(
        max_seq_length              = MAX_SEQ_LENGTH,
        packing                     = False,
        dataset_kwargs              = {"skip_prepare_dataset": True},
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs            = NUM_TRAIN_EPOCHS,
        warmup_ratio                = WARMUP_RATIO,
        learning_rate               = LEARNING_RATE,
        fp16                        = True,
        bf16                        = False,
        logging_steps               = 10,
        optim                       = "paged_adamw_8bit",
        weight_decay                = WEIGHT_DECAY,
        max_grad_norm               = MAX_GRAD_NORM,
        lr_scheduler_type           = "linear",
        seed                        = TRAIN_SEED,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
        remove_unused_columns       = False,
        ddp_find_unused_parameters  = False,
        gradient_checkpointing      = True,
        eval_strategy               = "no",
        save_strategy               = "no",
    ),
)

trainer_stats = trainer.train()

for i in range(torch.cuda.device_count()):
    used  = round(torch.cuda.max_memory_reserved(i) / 1024**3, 3)
    total = round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1)
    print(f"GPU {i}: peak VRAM = {used} GB / {total} GB ({100*used/total:.1f}%)")

print(f"Time   : {trainer_stats.metrics['train_runtime']:.0f} s  ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Loss   : {trainer_stats.metrics.get('train_loss', 'N/A'):.4f}")
# Persist the final adapter before releasing the trainer or starting inference.
model.peft_config["default"].revision = MODEL_REVISION
model.save_pretrained(LORA_SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(LORA_SAVE_DIR)
trainer.save_state()
Path("/kaggle/working/training_arguments.json").write_text(trainer.args.to_json_string())
Path("/kaggle/working/training_stats.json").write_text(json.dumps(trainer_stats.metrics, indent=2) + "\n")
hyperparameters = {
    "base_model": MODEL_NAME, "model_revision": MODEL_REVISION,
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
    "target_modules": "all-linear", "max_seq_length": MAX_SEQ_LENGTH,
    "batch_size": BATCH_SIZE, "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE, "epochs": NUM_TRAIN_EPOCHS,
    "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": MAX_GRAD_NORM, "seed": TRAIN_SEED,
    "max_chunk_chars": MAX_CHUNK_CHARS, "spacy_model": SPACY_MODEL,
    "max_negative_ratio": MAX_NEGATIVE_RATIO, "negative_sample_seed": NEGATIVE_SAMPLE_SEED,
    "system_prompt": SYSTEM_PROMPT, "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False, "dataset_fingerprint": DATASET_FINGERPRINT,
    "training_chunks": len(train_data),
}
Path("/kaggle/working/hyperparameters.json").write_text(json.dumps(hyperparameters, ensure_ascii=False, indent=2) + "\n")
adapter_files = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                 for p in sorted(Path(LORA_SAVE_DIR).iterdir()) if p.is_file()}
if "adapter_model.safetensors" not in adapter_files or "adapter_config.json" not in adapter_files:
    raise RuntimeError("The final LoRA adapter was not saved successfully.")
Path("/kaggle/working/adapter_manifest.json").write_text(json.dumps(adapter_files, indent=2) + "\n")
print(f"Final adapter and tokenizer saved to {LORA_SAVE_DIR}", flush=True)


## 7. Inference preparation

Release the trainer, configure the generation pipeline and list all 250 CardioCCC test documents.


In [ ]:
import gc
import time
import re
import json
import logging
import pandas as pd
from pathlib import Path
from transformers import pipeline

del trainer
gc.collect()
torch.cuda.empty_cache()

logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

model.eval()

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)

def build_ner_messages(text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Texto para analizar:\n{text}"},
    ]

txt_files = sorted(Path(TEXT_FILES_TEST_DIR).glob("*.txt"))
if MAX_TEST_FILES is not None:
    txt_files = txt_files[:MAX_TEST_FILES]

if len(txt_files) != 250:
    raise ValueError("This experiment requires all 250 CardioCCC test documents.")
processed_document_ids = [p.stem for p in txt_files]
Path("/kaggle/working/processed_documents.json").write_text(json.dumps(processed_document_ids, indent=2) + "\n")
print(f"Test documents to process: {len(txt_files)}")

## 8. Inference

Use the original sentence segmentation, decoding and character-offset alignment.


In [ ]:
pred_rows       = []
inference_times = []
total_files     = len(txt_files)
inference_errors = []

for i, txt_path in enumerate(txt_files):
    text = txt_path.read_text(encoding="utf-8")
    raw_mentions = []
    t0 = time.time()

    chunks = chunk_text_by_sentences(text, nlp_spacy)

    for chunk_idx, (chunk_text, chunk_char_start, chunk_char_end) in enumerate(chunks):
        prompt = tokenizer.apply_chat_template(
            build_ner_messages(chunk_text),
            tokenize=False,
            add_generation_prompt=True,
        )
        try:
            result = pipe(
                prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                max_length=None,
                do_sample=False,
            )[0]["generated_text"].strip()
        except RuntimeError as e:
            inference_errors.append({"filename": txt_path.stem, "chunk": chunk_idx, "error": str(e)})
            print(f"Inference error in {txt_path.name}, chunk {chunk_idx}: {e}", flush=True)
            if "out of memory" in str(e).lower():
                gc.collect()
                torch.cuda.empty_cache()
            continue

        if PRINT_RAW_OUTPUT:
            print(f"  [chunk {chunk_idx+1}/{len(chunks)} | chars {chunk_char_start}-{chunk_char_end}]")
            print(f"  RAW: {repr(result)}")

        search_cursors = {}
        for mention in [line.strip() for line in result.splitlines() if line.strip()]:
            pattern = re.escape(mention)
            cursor = search_cursors.get(mention, chunk_char_start)
            match = re.search(pattern, text[cursor:chunk_char_end], re.IGNORECASE)
            if not match:
                # Retry from the start of the chunk.
                match = re.search(pattern, text[chunk_char_start:chunk_char_end], re.IGNORECASE)
                if not match:
                    continue
                off0 = chunk_char_start + match.start()
                off1 = chunk_char_start + match.end()
            else:
                off0 = cursor + match.start()
                off1 = cursor + match.end()
            search_cursors[mention] = off1
            raw_mentions.append((text[off0:off1], off0, off1))

    t_infer = time.time() - t0
    inference_times.append(t_infer)

    raw_mentions.sort(key=lambda x: x[1])
    seen_offsets = set()
    deduped = []
    for span_text, off0, off1 in raw_mentions:
        if (off0, off1) not in seen_offsets:
            seen_offsets.add((off0, off1))
            deduped.append((span_text, off0, off1))

    for mark_idx, (span_text, off0, off1) in enumerate(deduped, 1):
        pred_rows.append({
            "filename": txt_path.stem,
            "mark":     f"T{mark_idx}",
            "label":    "ENFERMEDAD",
            "off0":     off0,
            "off1":     off1,
            "span":     span_text,
        })

    print(f"[{i+1}/{total_files}] {txt_path.name} | {t_infer:.2f}s | {len(chunks)} chunks | {len(deduped)} mentions")

pd.DataFrame(pred_rows, columns=["filename", "mark", "label", "off0", "off1", "span"]).to_csv(
    PREDICTIONS_TSV, sep="\t", index=False
)
print(f"\nPredictions saved to {PREDICTIONS_TSV}")
if inference_times:
    print(f"Mean time per document: {sum(inference_times)/len(inference_times):.2f}s")
Path("/kaggle/working/inference_stats.json").write_text(json.dumps({
    "documents": total_files, "total_seconds": sum(inference_times),
    "seconds_per_document": inference_times, "failed_chunks": inference_errors,
    "completed_successfully": not inference_errors,
}, indent=2) + "\n")
if inference_errors:
    raise RuntimeError("Inference had failed chunks; inspect inference_stats.json before using the predictions.")


## 9. Evaluation

Strict exact-offset metrics and overlap metrics at the original IoU thresholds.
All 250 test documents are included, even when the model predicts no mentions.


In [ ]:
import json
import pandas as pd


def prf(tp, fp, fn):
    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1


df_pred  = pd.read_csv(PREDICTIONS_TSV, sep="\t")
df_gs    = pd.read_csv(GS_TEST_TSV, sep="\t")
df_gs    = df_gs[df_gs["filename"].isin(processed_document_ids)]
if set(df_pred["filename"]) - set(processed_document_ids):
    raise ValueError("Predictions contain documents outside the external test set.")
if len(df_gs) != 7884:
    raise ValueError("The complete external gold standard must contain 7,884 mentions.")

set_gs   = set(zip(df_gs["filename"],   df_gs["label"],   df_gs["off0"],   df_gs["off1"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["off0"], df_pred["off1"]))

tp, fp, fn = len(set_gs & set_pred), len(set_pred - set_gs), len(set_gs - set_pred)
p, r, f1   = prf(tp, fp, fn)

report = {"Strict": {"tp": tp, "fp": fp, "fn": fn, "precision": round(p, 4), "recall": round(r, 4), "fscore": round(f1, 4)}}

print(f"── STRICT ──  P={p:.4f}  R={r:.4f}  F1={f1:.4f}  (TP={tp} FP={fp} FN={fn})\n")

thresholds = [0.0, 0.5, 0.8]
results    = {t: {"tp": 0, "fp": 0, "fn": 0} for t in thresholds}

for filename in processed_document_ids:
    gs_ints   = list(zip(df_gs[df_gs["filename"] == filename]["off0"],   df_gs[df_gs["filename"] == filename]["off1"]))
    pred_ints = list(zip(df_pred[df_pred["filename"] == filename]["off0"], df_pred[df_pred["filename"] == filename]["off1"]))

    iou_matrix = sorted(
        [(max(0, min(p1,g1) - max(p0,g0)) / (max(p1,g1) - min(p0,g0)), pi, gi)
         for pi, (p0, p1) in enumerate(pred_ints)
         for gi, (g0, g1) in enumerate(gs_ints)
         if max(p1,g1) - min(p0,g0) > 0 and min(p1,g1) - max(p0,g0) > 0],
        reverse=True,
    )

    for t in thresholds:
        matched_p, matched_g = set(), set()
        for iou, pi, gi in iou_matrix:
            if iou >= t and pi not in matched_p and gi not in matched_g:
                matched_p.add(pi); matched_g.add(gi)
        tp_t = len(matched_p)
        results[t]["tp"] += tp_t
        results[t]["fp"] += len(pred_ints) - tp_t
        results[t]["fn"] += len(gs_ints)   - tp_t

print("── IoU ──")
for t in thresholds:
    tp_t, fp_t, fn_t = results[t]["tp"], results[t]["fp"], results[t]["fn"]
    p_t, r_t, f1_t   = prf(tp_t, fp_t, fn_t)
    report[f"IoU >= {t}"] = {"tp": tp_t, "fp": fp_t, "fn": fn_t,
                              "precision": round(p_t, 4), "recall": round(r_t, 4), "fscore": round(f1_t, 4)}
    print(f"IoU >= {t}:  P={p_t:.4f}  R={r_t:.4f}  F1={f1_t:.4f}  (TP={tp_t} FP={fp_t} FN={fn_t})")

with open(EVAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)